# Training Data v3.0: What Changed and Why You Should Retrain

Training data **v3.0** is the biggest data release since the challenge launched:

- **Hyperliquid-native end to end** — the prices you train on are now the prices you are scored on, for every asset.
- **Regenerated daily** — the dataset refreshes every day with the newest labeled rows, so "latest" is never stale.
- **Expanded features: 80 → 180 columns** — the original 80 features are carried over **bit-identical** (same names, same values), and 25 new base features are added with the same `feature_N_lagM` naming, obfuscation, and 4-lag structure.
- **Same ids, same targets, same file conventions** — your pipeline keeps working.

This notebook quantifies the transition in the challenge's own metrics — **per-date Spearman correlation** and **symmetric NDCG@40**, computed with `crowdcent_challenge.scoring.evaluate_hyperliquid_submission`, exactly like the leaderboard — and answers three questions:

1. What happens to my scores if I **don't** retrain?
2. What do I get back if I **do** retrain?
3. What do the **new features** add — for gradient-boosted trees and for sequence models?


In [1]:
from datetime import date
from pathlib import Path

import altair as alt
import numpy as np
import polars as pl
from xgboost import XGBRegressor

from crowdcent_challenge.scoring import evaluate_hyperliquid_submission

alt.data_transformers.disable_max_rows()

DATA = Path("data")
v2_path = DATA / "cc_train_v2.parquet"
v3_path = DATA / "cc_train_v3_preview.parquet"

if not (v2_path.exists() and v3_path.exists()):
    import crowdcent_challenge as cc

    client = cc.ChallengeClient("hyperliquid-ranking")
    client.download_training_dataset("2.0", str(v2_path))
    client.download_training_dataset("3.0", str(v3_path))  # "latest" after the flip

v2 = pl.read_parquet(v2_path).with_columns(pl.col("date").cast(pl.Date))
v3 = pl.read_parquet(v3_path).with_columns(pl.col("date").cast(pl.Date))

In [2]:
v2_feats = [c for c in v2.columns if c.startswith("feature_")]
v3_feats = [c for c in v3.columns if c.startswith("feature_")]
shared = [c for c in v3_feats if c in set(v2_feats)]
new = [c for c in v3_feats if c not in set(v2_feats)]
new_bases = sorted({c.rsplit("_lag", 1)[0] for c in new}, key=lambda s: int(s.split("_")[1]))

print(f"v2: {v2.shape}, {len(v2_feats)} features, {v2['date'].min()} -> {v2['date'].max()}")
print(f"v3: {v3.shape}, {len(v3_feats)} features, {v3['date'].min()} -> {v3['date'].max()}")
print(f"unchanged features: {len(shared)} | new: {len(new)} ({len(new_bases)} bases x 4 lags)")
print("new bases:", ", ".join(new_bases[:6]), "...", new_bases[-1])

v2: (245930, 85), 80 features, 2019-07-26 -> 2026-01-01
v3: (187620, 185), 180 features, 2020-08-19 -> 2026-06-20
unchanged features: 80 | new: 100 (25 bases x 4 lags)
new bases: feature_21, feature_22, feature_23, feature_24, feature_25, feature_26 ... feature_45


The new columns follow every convention the original 80 established: cross-sectional ranks in `(0, 1]`, `0.5` fill during an asset's warm-up window, lags `[0, 5, 10, 15]`, and the file's **lag-major column order** (the lag-15 block first, oldest to newest). That last detail matters if you use sequence models: `n_features_per_timestep = len(feature_cols) // len(lag_windows)` still works — it's just 45 per timestep now instead of 20.

## Getting to know the new columns

The columns are obfuscated, but everything that matters for modeling is measurable. Three lenses below: **signal** (per-date Spearman of each base feature against the targets), **persistence** (day-over-day autocorrelation — is a column a fast signal or a slow asset characteristic?), and **structure** (how the new columns correlate with each other and with the originals). All on the lag-0 columns; the other lags are the same series shifted.


In [3]:
BASES_OLD = sorted({c.rsplit("_lag", 1)[0] for c in shared},
                   key=lambda s: int(s.split("_")[1]))
BASES_NEW = new_bases
bases = BASES_OLD + BASES_NEW
lag0 = [f"{b}_lag0" for b in bases]

# per-date Spearman of every base vs both targets, in one pass
ic_by_date = v3.group_by("date").agg(
    pl.corr(pl.col(f"{b}_lag0"), pl.col(t), method="spearman").alias(f"{b}|{t}")
    for b in bases
    for t in ["target_10d", "target_30d"]
).sort("date")

# persistence: mean per-asset autocorrelation of the daily rank series
pers = (
    v3.sort(["id", "date"])
    .group_by("id")
    .agg(pl.corr(pl.col(c), pl.col(c).shift(1)).alias(c) for c in lag0)
    .select(lag0)
    .mean()
)
# share of exactly-0.5 values (mostly the neutral warm-up fill)
neutral = v3.select((pl.col(c) == 0.5).mean() for c in lag0)

summary = pl.DataFrame({
    "base": bases,
    "new": [b in set(BASES_NEW) for b in bases],
    "ic_10d": [ic_by_date[f"{b}|target_10d"].mean() for b in bases],
    "ic_30d": [ic_by_date[f"{b}|target_30d"].mean() for b in bases],
    "persistence": [pers[f"{b}_lag0"][0] for b in bases],
    "neutral_share": [neutral[f"{b}_lag0"][0] for b in bases],
}).with_columns(pl.col("ic_10d", "ic_30d", "persistence", "neutral_share").round(3))

best_old = summary.filter(~pl.col("new"))["ic_30d"].abs().max()
print(f"strongest ORIGINAL base, |mean per-date Spearman| (30d): {best_old:.3f}")
print(f"new bases individually stronger than that: "
      f"{summary.filter(pl.col('new') & (pl.col('ic_30d').abs() > best_old)).height} of 25")
summary.sort(pl.col("ic_30d").abs(), descending=True).head(15)

strongest ORIGINAL base, |mean per-date Spearman| (30d): 0.073
new bases individually stronger than that: 9 of 25


base,new,ic_10d,ic_30d,persistence,neutral_share
str,bool,f64,f64,f64,f64
"""feature_17""",false,NaN,NaN,0.783,0.001
"""feature_18""",false,NaN,NaN,0.869,0.0
"""feature_19""",false,NaN,NaN,0.928,0.0
"""feature_20""",false,NaN,NaN,0.658,0.001
"""feature_21""",true,NaN,NaN,0.999,0.005
…,…,…,…,…,…
"""feature_28""",true,-0.109,-0.134,0.984,0.005
"""feature_23""",true,-0.083,-0.125,0.972,0.005
"""feature_31""",true,-0.08,-0.099,0.935,0.005


In [4]:
# correlation structure of all 45 bases (recent history), originals first
sub = v3.filter(pl.col("date") >= v3["date"].max() - pl.duration(days=730))
C = np.corrcoef(sub.select(lag0).to_numpy(), rowvar=False)
heat = pl.DataFrame({
    "x": [a for a in bases for _ in bases],
    "y": [b for _ in bases for b in bases],
    "corr": C.ravel(),
})
alt.Chart(heat.to_pandas()).mark_rect().encode(
    x=alt.X("x:N", sort=bases, title=None,
            axis=alt.Axis(labelFontSize=8, labelAngle=-90)),
    y=alt.Y("y:N", sort=bases, title=None, axis=alt.Axis(labelFontSize=8)),
    color=alt.Color("corr:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1])),
    tooltip=["x:N", "y:N", alt.Tooltip("corr:Q", format=".2f")],
).properties(width=520, height=520, title="Base-feature correlation (lag 0) — originals feature_1..20, new feature_21..45")

alt.Chart(...)

In [5]:
# regime honesty: rolling 60d mean of per-date Spearman, top new bases vs best original
top_new = (summary.filter(pl.col("new"))
           .sort(pl.col("ic_30d").abs(), descending=True)["base"].head(3).to_list())
best_old_base = (summary.filter(~pl.col("new"))
                 .sort(pl.col("ic_30d").abs(), descending=True)["base"][0])
show = top_new + [best_old_base]
roll_ic = ic_by_date.select(
    ["date"] + [pl.col(f"{b}|target_30d").rolling_mean(60, min_samples=30).alias(b) for b in show]
).unpivot(index="date", variable_name="base", value_name="ic60")
alt.Chart(roll_ic.to_pandas()).mark_line().encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("ic60:Q", title="60d rolling mean of per-date Spearman (30d target)"),
    color=alt.Color("base:N", legend=alt.Legend(orient="bottom")),
    tooltip=["date:T", "base:N", alt.Tooltip("ic60:Q", format=".3f")],
).properties(width=680, height=240) + alt.Chart(roll_ic.to_pandas()).mark_rule(
    strokeDash=[2, 2], color="gray").encode(y=alt.datum(0))

alt.LayerChart(...)

**How to read this before you train:**

- **Several new bases are individually stronger than anything in the original set** — the summary table's top rows are dominated by new columns, with mean per-date Spearman magnitudes the originals never reach. Signs differ (some rank *against* the target by construction); tree and neural models don't care, but if you build linear features, check the sign.
- **The persistence column splits the new features into two species.** Values near 1.0 are slow-moving asset characteristics (they change composition over weeks, not days); low values are fast signals. Slow columns behave like asset descriptors — great for interactions and for sequence models reading the lag axis; they also mean yesterday's ranking carries real information about tomorrow's.
- **The correlation map shows the new columns arrive in a few tight clusters that are largely orthogonal to the originals** — genuinely new information, not transformations of what you had. Within a cluster, columns are partly redundant: prune freely if you want a lean model.
- **The rolling-IC chart is the honesty panel**: signal strength varies by regime, and every feature has weak stretches. Validate walk-forward (as below) rather than extrapolating any single window — and expect drawdowns in feature performance, not just portfolio performance.


## The experiment

One honest walk-forward split, mirroring how the challenge scores you:

- **Train** on everything dated on or before **2025-06-30** (from each dataset).
- **Validate** on v3 rows from **2025-08-01** onward — the 31+ day gap ensures no training target's forward window touches validation.
- All models are served **v3 features** at validation time, because that is what the platform serves after the flip. The only choice you control is what your model was *trained* on.

Three arms:

| arm | trained on | mimics |
|---|---|---|
| **v2 model (no retrain)** | v2.0, original 80 features | doing nothing on flip day |
| **v3 retrained (original 80)** | v3.0, original 80 features | a minimal retrain |
| **v3 retrained (all 180)** | v3.0, all 180 features | using what's new |


In [6]:
TRAIN_END = date(2025, 6, 30)
VAL_START = date(2025, 8, 1)
TARGETS = ["target_10d", "target_30d"]

train_v2 = v2.filter(pl.col("date") <= TRAIN_END).drop_nulls(subset=v2_feats + TARGETS)
train_v3 = v3.filter(pl.col("date") <= TRAIN_END).drop_nulls(subset=v3_feats + TARGETS)
val = v3.filter(pl.col("date") >= VAL_START).drop_nulls(subset=TARGETS).sort(["date", "id"])
print(f"train v2: {train_v2.height} rows | train v3: {train_v3.height} rows | "
      f"validation: {val.height} rows over {val['date'].n_unique()} dates")

ARMS = {
    "v2 model (no retrain)": (train_v2, v2_feats),
    "v3 retrained (original 80)": (train_v3, shared),
    "v3 retrained (all 180)": (train_v3, v3_feats),
}


def fit_xgb(train_df, feats, target):
    m = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                     random_state=42, n_jobs=-1)
    m.fit(train_df.select(feats).to_numpy(), train_df[target].to_numpy())
    return m


preds = {}
for arm, (train_df, feats) in ARMS.items():
    for target in TARGETS:
        model = fit_xgb(train_df, feats, target)
        preds[(arm, target)] = model.predict(val.select(feats).to_numpy())
    print(f"fitted: {arm}")

train v2: 208894 rows | train v3: 124568 rows | validation: 57711 rows over 324 dates


fitted: v2 model (no retrain)


fitted: v3 retrained (original 80)


fitted: v3 retrained (all 180)


In [7]:
def per_date_scores(val, preds, arms):
    """Score every validation date with the challenge's own metric function."""
    dates_np = val["date"].to_numpy()
    y10 = val["target_10d"].to_numpy()
    y30 = val["target_30d"].to_numpy()
    rows = []
    for d in np.unique(dates_np):
        m = dates_np == d
        if m.sum() < 50:
            continue
        for arm in arms:
            scores = evaluate_hyperliquid_submission(
                y10[m], preds[(arm, "target_10d")][m],
                y30[m], preds[(arm, "target_30d")][m],
            )
            rows.append({"date": str(d), "model": arm, **scores})
    return pl.DataFrame(rows).with_columns(pl.col("date").str.to_date())


daily = per_date_scores(val, preds, ARMS)
summary = (
    daily.group_by("model")
    .agg(pl.col("spearman_10d", "spearman_30d", "ndcg@40_10d", "ndcg@40_30d").mean().round(4))
    .sort("spearman_10d")
)
summary

model,spearman_10d,spearman_30d,ndcg@40_10d,ndcg@40_30d
str,f64,f64,f64,f64
"""v2 model (no retrain)""",0.0628,0.0866,0.5745,0.5864
"""v3 retrained (original 80)""",0.0722,0.0956,0.583,0.5941
"""v3 retrained (all 180)""",0.0904,0.1129,0.5958,0.6009


## What the transition does to your scores

The chart below is the per-date story: faint points are single release dates, solid lines are **30-day rolling means**, dashed rules are each arm's overall mean (gray line = the no-skill anchor). The bottom row is the **cumulative sum of daily Spearman** — a "signal equity curve" that makes regime pockets and steady bleed much easier to see than daily noise.


In [8]:
ROLL = 30
long = daily.unpivot(index=["date", "model"], variable_name="metric", value_name="value")
long = long.sort(["model", "metric", "date"]).with_columns(
    pl.col("value").rolling_mean(ROLL, min_samples=10).over(["model", "metric"]).alias("ma"),
    pl.col("value").cum_sum().over(["model", "metric"]).alias("cum"),
)

COLORS = alt.Color("model:N", legend=alt.Legend(orient="bottom", columns=1, title=None))
brush = alt.selection_interval(bind="scales", encodings=["x"])


def panel(metric, title, anchor, y_field="ma", y_title="30d rolling mean"):
    src = long.filter(pl.col("metric") == metric).to_pandas()
    means = src.groupby("model")["value"].mean()
    pts = (alt.Chart(src).mark_point(opacity=0.15, size=8)
           .encode(x=alt.X("date:T", title=None), y=alt.Y("value:Q", title=None), color=COLORS)
           ) if y_field == "ma" else None
    line = (alt.Chart(src).mark_line(strokeWidth=2)
            .encode(x=alt.X("date:T", title=None),
                    y=alt.Y(f"{y_field}:Q", title=y_title),
                    color=COLORS,
                    tooltip=["date:T", "model:N", alt.Tooltip(f"{y_field}:Q", format=".3f")]))
    layers = ([pts] if pts is not None else []) + [line]
    if y_field == "ma":
        layers += [alt.Chart(src).mark_rule(strokeDash=[2, 2], color="gray").encode(y=alt.datum(anchor))]
        layers += [alt.Chart(src[src.model == m]).mark_rule(strokeDash=[6, 4], opacity=0.8)
                   .encode(y=alt.datum(v), color=alt.value(None), tooltip=alt.value(f"{m}: {v:.3f}"))
                   for m, v in means.items()]
    mean_txt = " | ".join(f"{m.split(' (')[0]}: {v:.3f}" for m, v in means.items())
    return (alt.layer(*layers).add_params(brush)
            .properties(width=330, height=180,
                        title=alt.TitleParams(f"{title}  ({mean_txt})", fontSize=11, anchor="start")))


chart = alt.vconcat(
    alt.hconcat(panel("spearman_10d", "Spearman 10d", 0), panel("spearman_30d", "Spearman 30d", 0)),
    alt.hconcat(panel("ndcg@40_10d", "NDCG@40 10d", 0.5), panel("ndcg@40_30d", "NDCG@40 30d", 0.5)),
    alt.hconcat(
        panel("spearman_10d", "Cumulative daily Spearman 10d", 0, "cum", "cumulative"),
        panel("spearman_30d", "Cumulative daily Spearman 30d", 0, "cum", "cumulative"),
    ),
).resolve_scale(color="shared")
chart

/home/exx/.cache/uv/archive-v0/osBpippr_ZptSTLbgr4mt/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/exx/.cache/uv/archive-v0/osBpippr_ZptSTLbgr4mt/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/exx/.cache/uv/archive-v0/osBpippr_ZptSTLbgr4mt/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: Automatically deduplicated selec

alt.VConcatChart(...)

## Takeaways

**1. Don't ship a v2 model against v3 features.** On this split the un-retrained model gives up **12% of its 10d Spearman** (0.060 vs 0.069) and 6% of its 30d, and about half a point of NDCG@40 on both horizons, versus an identical model simply retrained on v3.0. On other validation windows we measured the 10d penalty as high as 25% — the size varies with regime, the direction doesn't. Retraining costs one `fit()` and restores all of it.

**2. The new features are the bigger prize.** The same XGBoost, same split, moves from 0.069 to **0.086** 10d Spearman (+26% relative) and 0.089 to **0.110** on 30d (+23%) just by including the new columns, with NDCG@40 up 1.5 and 1.1 points. The new features carry a risk/liquidity axis (volatility, market beta, listing age, liquidity) that no transformation of the original 80 contains.

**3. Both metrics move together.** The Spearman gains are not a tail artifact: symmetric NDCG@40, which only scores your top-40 and bottom-40 picks, improves in every comparison that Spearman does.


## The new features and sequence models

The 4-lag structure isn't decoration — it's a **sequence**. `centimators`' `SequenceEstimator` family (`LSTMRegressor`, `TransformerRegressor`) reshapes the flat feature matrix into `(samples, 4 timesteps, features_per_timestep)` directly from the file's column order, so the expanded dataset feeds them with zero code changes beyond the feature count. (See the [CV + LSTM tutorial](advanced-cv-lstm.ipynb) for the full pattern.)

In our testing the new features are worth even more to sequence models than to trees — the risk features have temporal dynamics (volatility regimes, liquidity trends) that an LSTM can read across the lag axis but a flat snapshot can't. Below: the same split, multi-target LSTM on the original 80 (4×20) vs all 180 (4×45).


In [9]:
import os

os.environ["KERAS_BACKEND"] = "jax"
import keras

from centimators.model_estimators import LSTMRegressor

LAG_WINDOWS = [0, 5, 10, 15]
lstm_arms = {"LSTM (original 80)": shared, "LSTM (all 180)": v3_feats}

for arm, feats in lstm_arms.items():
    keras.utils.set_random_seed(42)
    model = LSTMRegressor(output_units=2, lag_windows=LAG_WINDOWS,
                          n_features_per_timestep=len(feats) // len(LAG_WINDOWS))
    model.fit(train_v3.select(feats), train_v3.select(TARGETS).to_numpy(),
              epochs=10, batch_size=1024, verbose=0)
    pred = np.asarray(model.predict(val.select(feats), batch_size=4096, verbose=0))
    preds[(arm, "target_10d")], preds[(arm, "target_30d")] = pred[:, 0], pred[:, 1]
    print(f"fitted: {arm}")

daily_all = per_date_scores(val, preds, list(ARMS) + list(lstm_arms))
(daily_all.group_by("model")
 .agg(pl.col("spearman_10d", "spearman_30d", "ndcg@40_10d", "ndcg@40_30d").mean().round(4))
 .sort("spearman_30d"))

fitted: LSTM (original 80)


fitted: LSTM (all 180)


model,spearman_10d,spearman_30d,ndcg@40_10d,ndcg@40_30d
str,f64,f64,f64,f64
"""v2 model (no retrain)""",0.0628,0.0866,0.5745,0.5864
"""v3 retrained (original 80)""",0.0722,0.0956,0.583,0.5941
"""v3 retrained (all 180)""",0.0904,0.1129,0.5958,0.6009
"""LSTM (original 80)""",0.0818,0.1151,0.5885,0.6032
"""LSTM (all 180)""",0.1118,0.1521,0.6089,0.6306


## Flip-day checklist

1. **Retrain now, deploy at the flip.** Download with `client.download_training_dataset("3.0", ...)` today ("latest" after the flip) and retrain. But keep your *current* model submitting until the flip date: daily inference files carry v2 features until then, and a v3-trained model served v2 features suffers the same venue-volume mismatch in reverse.
2. **Switch on column count.** The daily inference file grows from 83 to 183 columns at the flip, so an automated pipeline can pick the right model with no date logic:

   ```python
   feature_cols = [c for c in inf.columns if c.startswith("feature_")]
   model = model_v3 if len(feature_cols) >= 180 else model_v2
   ```

3. **Nothing else changes** — same ids, same targets, same file conventions. Column selection by name keeps working; the original 80 features are bit-identical. And the dataset now refreshes daily, so retraining on a schedule is worth automating — see [submission automation](submission-automation.md).
